# LegalSearchTool 래핑과 ToolResult 검증 샘플 (2026-11-05)

## 목적 (11/5 유종혁 담당 · 공통계약 v2.0)
1. 기존 `run()`을 `legal_search(ToolInput) -> ToolResult`로 감싼다 (`src/rag/prevention_legal/tool.py`).
2. `ToolResult` 상태 규칙(OK / PARTIAL / NO_EVIDENCE / ERROR)과 Finding↔Citation 연결을 검사하는 `validate_tool_result`를 만든다.
3. 근거와 신고 주체를 보존하는지 샘플로 확인한다. 오류와 근거 없음을 구분하는지도 확인한다.

## 상태 대응
| run() 상태 | ToolResult 상태 |
|---|---|
| ANSWERED | OK |
| PARTIAL | PARTIAL |
| INSUFFICIENT_EVIDENCE | NO_EVIDENCE |
| 실행 예외 | ERROR (findings·citations 비움) |

In [1]:
from rag.prevention_legal.tool import legal_search, validate_tool_result
from rag.prevention_legal.run import SAMPLE_QUESTION
# 성홍열 질문 목록 (고정 답 목록은 2026-11-06 삭제. 질문만 남기고 검색 카드로 답한다)
SCARLET_CASES = [
    {"question": "성홍열은 감염병예방법상 몇 급감염병이며, 신고는 언제까지 해야 하나요?"},
    {"question": "성홍열 환자가 발생했을 때 학교 관리인은 누구에게 어떻게 신고해야 하나요?"},
    {"question": "성홍열 학교 대응 절차를 지침에 따라 자세히 알려 주세요."},
]

BOTH = ["kdca_measles_2026", "moleg_infectious_disease_act_21521"]
MEASLES_ONLY = ["kdca_measles_2026"]

def call(question, allowed):
    tin = {"question": question, "task": "legal_search", "region": "NATIONAL", "allowed_document_ids": allowed}
    res = legal_search(tin)
    validate_tool_result(res, tin)   # 검증 실패하면 여기서 오류
    return res

cases = [
    ("홍역 샘플 (둘 다 허용)", SAMPLE_QUESTION, BOTH),
    ("홍역 샘플 (홍역 지침만 허용)", SAMPLE_QUESTION, MEASLES_ONLY),
    ("성홍열 Q31 급수·신고 시기", SCARLET_CASES[0]["question"], BOTH),
    ("성홍열 Q32 학교 관리인", SCARLET_CASES[1]["question"], BOTH),
    ("성홍열 Q33 학교 대응 절차 (근거 부족)", SCARLET_CASES[2]["question"], BOTH),
    ("샘플 밖 질문", "다른 질문입니다", BOTH),
]
for name, q, allowed in cases:
    r = call(q, allowed)
    print(f"{name} | {r['status']} | Finding {len(r['findings'])} | Citation {len(r['citations'])}")

In [2]:
# 오류 경로: 입력에 필수 키가 없으면 실패를 근거 없음으로 바꾸지 않고 ERROR 로 돌려준다
bad_input = {"question": SAMPLE_QUESTION, "task": "legal_search", "region": "NATIONAL"}   # allowed_document_ids 누락
r = legal_search(bad_input)
print("오류 경로 |", r["status"], "| warnings:", r["warnings"])
validate_tool_result(r, {"question": SAMPLE_QUESTION, "task": "legal_search", "region": "NATIONAL", "allowed_document_ids": BOTH})
print("ERROR 결과 검증 통과")

오류 경로 | ERROR | warnings: ['Tool 실행 실패: KeyError']
ERROR 결과 검증 통과


In [3]:
# 검증이 잡아야 하는 잘못된 결과 (음성 검사)
tin = {"question": SAMPLE_QUESTION, "task": "legal_search", "region": "NATIONAL", "allowed_document_ids": BOTH}
good = legal_search(tin)
tests = {
    "OK인데 근거 없음": {**good, "status": "OK", "findings": [], "citations": []},
    "Finding 이 없는 Citation 을 가리킴": {**good, "findings": [{**good["findings"][0], "chunk_ids": ["없는:p0001:b01"]}]},
    "NO_EVIDENCE 인데 근거 있음": {**good, "status": "NO_EVIDENCE"},
    "허용되지 않은 문서 인용": {**good, "citations": [{**good["citations"][0], "document_id": "other_doc"}]},
}
for name, bad in tests.items():
    try:
        validate_tool_result(bad, tin)
        print("놓침 (문제):", name)
    except ValueError as e:
        print("검출 ✅:", name, "|", e)

검출 ✅: OK인데 근거 없음 | OK 는 Finding 1개 이상과 Citation 이 필요합니다
검출 ✅: Finding 이 없는 Citation 을 가리킴 | Finding 이 없는 Citation 을 가리킵니다: ['없는:p0001:b01']
검출 ✅: NO_EVIDENCE 인데 근거 있음 | NO_EVIDENCE 는 findings·citations 가 비어 있어야 합니다
검출 ✅: 허용되지 않은 문서 인용 | 허용되지 않은 문서의 Citation 입니다
